# 2.ipynb — entity + location join, step by step

Nine steps, one per section. Each step posts something and prints what came back.

**Before running:**
1. `python run.py` — API on http://localhost:8000
2. Run `reset_db.ipynb` for a clean slate (every POST is stored, so re-runs pile up history rows).

Every POST to `/entities/process` or `/entities/inputlocation` runs the validation
rules against the **join** of `entities` + `location` on `systemCode` +
`businessEntityCode`. The built-in `JOIN-COMPLETENESS` rule reports a missing
side: `status -11` = no entity yet, `status -12` = no location yet.


## Step 1 — Test the connection to the API

In [9]:
import json
import requests
import pandas as pd

BASE_URL = "http://localhost:8000"


def post(path, body):
    """POST and return the parsed response (raises on error)."""
    r = requests.post(f"{BASE_URL}{path}", json=body)
    if r.status_code >= 400:
        raise RuntimeError(f"{r.status_code}: {r.text}")
    return r.json()


spec = requests.get(f"{BASE_URL}/openapi.json").json()
print("connected:", BASE_URL)
print("routes:", sorted(spec["paths"]))


connected: http://localhost:8000
routes: ['/entities', '/entities/historical', '/entities/inputlocation', '/entities/joined', '/entities/locations', '/entities/locations/historical', '/entities/process', '/entities/upload-file', '/entities/validation-results', '/entities/{business_entity_code}', '/responsible-contacts', '/validation-rules']


## Step 2 — Insert 2 validation rules

One rule reads a field that only the **entity** side has (`EOID`), the other a
field that only the **location** side has (`country`). Both are `Exist` checks,
so together they show the rules really are evaluated against the join.


In [10]:
entity_rule = {
    "rule_id": "NB2-ENTITY-EOID",
    "validation_type": "Entity - Completeness",
    "description": "EOID must exist",
    "severity": "Critical",
    "on_exception_status": -21,
    "on_exception_description": "EOID is missing",
    "conditions": [
        {"role": "Target", "field": "EOID", "data_type": "Exist"}
    ],
}

location_rule = {
    "rule_id": "NB2-LOCATION-COUNTRY",
    "validation_type": "Location Master Data - Completeness",
    "description": "country must exist",
    "severity": "Critical",
    "on_exception_status": -22,
    "on_exception_description": "country is missing",
    "conditions": [
        {"role": "Target", "field": "country", "data_type": "Exist"}
    ],
}

print(json.dumps(post("/validation-rules", entity_rule), indent=2))
print(json.dumps(post("/validation-rules", location_rule), indent=2))

# the rules now in the database
display(pd.DataFrame(requests.get(f"{BASE_URL}/validation-rules").json()["rules"]))


{
  "rules_written": [
    "NB2-ENTITY-EOID"
  ],
  "conditions_written": 1,
  "checks_written": [],
  "logic_written": [],
  "generated_ids": [],
  "warnings": []
}
{
  "rules_written": [
    "NB2-LOCATION-COUNTRY"
  ],
  "conditions_written": 1,
  "checks_written": [],
  "logic_written": [],
  "generated_ids": [],
  "warnings": []
}


,rule_id,validation_type,function_name,check_id,description,severity,active,on_exception_status,on_exception_description
0,API-ROMANIZE-NAME,Location Master Data - Romanization,None,None,Location name must be Latin/ASCII only,Warning,Y,-1,Location name is not romanized
1,NB2-ENTITY-EOID,Entity - Completeness,None,None,EOID must exist,Critical,Y,-21,EOID is missing
2,NB2-LOCATION-COUNTRY,Location Master Data - Completeness,None,None,country must exist,Critical,Y,-22,country is missing


## Step 3 — Insert 1 location

No entity exists for `SYS-A / BE-001` yet, so expect
`awaiting_counterpart: 1` and `JOIN-COMPLETENESS` failing with **status -11**
(entity data is missing).


In [11]:
location_1 = {
    "systemCode": "SYS-A",
    "Code": "BE-001",
    "Name": "Warehouse One",
    "Type": "WHS",
    "Country": "UK",
    "City": "London",
    "Zip": "E1 6AN",
}

print(json.dumps(post("/entities/inputlocation", location_1), indent=2))


{
  "total": 1,
  "inserted": 1,
  "new_records": 0,
  "updated_records": 1,
  "joined": 1,
  "awaiting_counterpart": 0,
  "details": [
    {
      "status": "inserted",
      "id": 22,
      "new_record": false,
      "entity_hash": "668bae11288e4ca931d34fa811bf20bb7870342e124c985e1c471f5d436adb06",
      "systemCode": "SYS-A",
      "businessEntityCode": "BE-001",
      "joined": true
    }
  ]
}


## Step 4 — Insert 1 entity with the same systemCode + businessEntityCode

Same key as step 3, so the two sides now join: expect `joined: 1` and
`JOIN-COMPLETENESS` passing. Both rules from step 2 can see their field.


In [12]:
entity_1 = {
    "mappingsInformation": [
        {
            "systemCode": "SYS-A",
            "businessEntityCode": "BE-001",
            "EOID": "EO-0001",
            "FID": "FID-0001",
            "SGLN": "urn:epc:id:sgln:0614141.00001.0",
        }
    ]
}

print(json.dumps(post("/entities/process", entity_1), indent=2))


{
  "total": 1,
  "inserted": 1,
  "new_records": 0,
  "updated_records": 1,
  "joined": 1,
  "awaiting_counterpart": 0,
  "details": [
    {
      "status": "inserted",
      "id": 49,
      "new_record": false,
      "entity_hash": "668bae11288e4ca931d34fa811bf20bb7870342e124c985e1c471f5d436adb06",
      "systemCode": "SYS-A",
      "businessEntityCode": "BE-001",
      "joined": true
    }
  ]
}


## Step 5 — Insert more than 1 entity

Two new entities, neither of which has a location yet: expect
`awaiting_counterpart: 2` and **status -12** (location data is missing).


In [13]:
entities_many = {
    "mappingsInformation": [
        {
            "systemCode": "SYS-A",
            "businessEntityCode": "BE-002",
            "EOID": "EO-0002",
            "FID": "FID-0002",
        },
        {
            "systemCode": "SYS-A",
            "businessEntityCode": "BE-003",
            "EOID": "EO-0003",
            "FID": "FID-0003",
        },
    ]
}

print(json.dumps(post("/entities/process", entities_many), indent=2))


{
  "total": 2,
  "inserted": 2,
  "new_records": 0,
  "updated_records": 2,
  "joined": 0,
  "awaiting_counterpart": 2,
  "details": [
    {
      "status": "inserted",
      "id": 50,
      "new_record": false,
      "entity_hash": "28755534b8b0ba9405dc6740f02b37abc10a6483124d3a1653ea0deb9749f2e0",
      "systemCode": "SYS-A",
      "businessEntityCode": "BE-002",
      "joined": false
    },
    {
      "status": "inserted",
      "id": 51,
      "new_record": false,
      "entity_hash": "45d93593090845f9d17f8bbc3022f76554f03fb43e33dc9ee6f09b6374d33178",
      "systemCode": "SYS-A",
      "businessEntityCode": "BE-003",
      "joined": false
    }
  ]
}


## Step 6 — Insert more than 1 location, one without an entity

`BE-002` matches the entity posted in step 5, `BE-999` matches nothing: expect
`joined: 1` and `awaiting_counterpart: 1` (**status -11** for `BE-999`).


In [14]:
locations_with_orphan = [
    {
        "systemCode": "SYS-A",
        "Code": "BE-002",
        "Name": "Warehouse Two",
        "Country": "GB",
        "City": "Manchester",
    },
    {
        "systemCode": "SYS-A",
        "Code": "BE-999",
        "Name": "Orphan Site",
        "Country": "IE",
        "City": "Dublin",
    },
]

print(json.dumps(post("/entities/inputlocation", locations_with_orphan), indent=2))


{
  "total": 2,
  "inserted": 2,
  "new_records": 2,
  "updated_records": 0,
  "joined": 1,
  "awaiting_counterpart": 1,
  "details": [
    {
      "status": "inserted",
      "id": 23,
      "new_record": true,
      "entity_hash": "28755534b8b0ba9405dc6740f02b37abc10a6483124d3a1653ea0deb9749f2e0",
      "systemCode": "SYS-A",
      "businessEntityCode": "BE-002",
      "joined": true
    },
    {
      "status": "inserted",
      "id": 24,
      "new_record": true,
      "entity_hash": "4cc342ef3822fb27299cfd4a16568bbfa1838d38487044e58eb730ae1d783867",
      "systemCode": "SYS-A",
      "businessEntityCode": "BE-999",
      "joined": false
    }
  ]
}


## Step 7 — Insert more than 1 location

`BE-003` matches the entity from step 5. `BE-001` is posted **again** with a new
city — a second row in `historical_location`, replacing the single row in
`location`. The last cell shows both tables.


In [ ]:
locations_many = [
    {
        "systemCode": "SYS-A",
        "Code": "BE-003",
        "Name": "Warehouse Three",
        "Country": "UK",
        "City": "Leeds",
    },
    {
        "systemCode": "SYS-A",
        "Code": "BE-001",
        "Name": "Warehouse One",
        "Country": "UK",
        "City": "Bristol",
    },
]

print(json.dumps(post("/entities/inputlocation", locations_many), indent=2))


In [ ]:
# BE-001 appears twice in the history, once (Bristol) in the latest table
display(pd.DataFrame(requests.get(f"{BASE_URL}/entities/locations/historical").json()))
display(pd.DataFrame(requests.get(f"{BASE_URL}/entities/locations").json()))


## Step 8 — Insert more than 1 entity, one without a location

`BE-004` has no location, `BE-999` finally gives the orphan location from step 6
its entity: expect `joined: 1` and `awaiting_counterpart: 1` (**status -12** for
`BE-004`).


In [ ]:
entities_with_orphan = {
    "mappingsInformation": [
        {
            "systemCode": "SYS-A",
            "businessEntityCode": "BE-004",
            "EOID": "EO-0004",
        },
        {
            "systemCode": "SYS-A",
            "businessEntityCode": "BE-999",
            "EOID": "EO-0999",
        },
    ]
}

print(json.dumps(post("/entities/process", entities_with_orphan), indent=2))


## Step 9 — Check the validation results

Three views:
1. the join itself — which records have both sides,
2. the latest outcome per record and rule,
3. every validation row, newest first.


In [8]:
# 1. the join the rules are evaluated against
display(pd.DataFrame(requests.get(f"{BASE_URL}/entities/joined").json()))


,entity_hash,systemCode,businessEntityCode,join_status,entity_row_id,EOID,FID,UKEOID,UKFID,SGLN,entity_updated_at,location_row_id,name,type,alternate_code,country,city,zip,location_updated_at
0,a3cc2721bd8fa7cdf22294cc45b2a42de4533bbaebb9c2...,ACD123C,BGN1,missing_location,30,LEBGR1e003Tp4N3J,LEBGR1f0asd0asldasldk5OF84yaBGN1,4068,,urn:epc:id:sgln:1233367.11164.0,2026-09-22 03:41:05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,71fd22990c4a2aa70d0373eae0aef9a12b4dcd2e1296f3...,ACDC,BG01,complete,3,LEBGR1e003Tp4N3J,LEBGR1f005OF8asda4ya,4276,hQsJAyxU94KFfK9,urn:epc:id:sgln:1233367.11754.0,2026-09-22 03:37:22,3.0,Sofia Logistics Hub,Warehouse,BG-SOF-01,BG,Sofia,1000,2026-09-24 08:11:05
2,dc46c2661078a6a396cbe3f3433a9991cc4d58e4b4d27d...,ACDC,BG02,complete,4,LEBGR1e003Tasdp4N3J,LEBGR1f005OF84ya,2745,8GrI42lHnslSc05,urn:epc:id:sgln:1233367.10054.0,2026-09-22 03:37:22,4.0,Hamburg Cross-Dock,Warehouse,DE-HAM-02,DE,Hamburg,20095,2026-09-24 08:11:05
3,b9af7688dbfc47805a424eebc8bdf35bf1ffd5334096a5...,ACDC,BGN1,complete,5,LEBGR1e003Tp4N3J,LEBGR1f0asd05OF84yaBGN1,4068,Vw2QuxVutMftJ1V,urn:epc:id:sgln:1233367.11164.0,2026-09-22 03:37:22,5.0,Tokyo Bay Depot,Distribution,JP-TYO-1,JP,Tokyo,135-0064,2026-09-24 08:11:05
4,b189ac0ea2bab10e39a2764916b3d2536272016a351c1d...,ACDC,DOCS99,complete,2,QCLUXXe00000023,QCLUXXf00000023000018,2429,0ims9SUpP0txasdavB2,urn:epc:id:sgln:7173736.79084.0,2026-09-22 03:37:22,2.0,Shah Alam Plant,Manufacturing,MY-SA-99,MY,Shah Alam,40150,2026-09-24 08:11:05
5,3f94a784fb62851b506b0c14588449b073a4e8604c0edf...,ACDasC,DOCS99,missing_location,27,QCLUXXe00000023,QCLUXXf00000023000018,,0ims9SUpP0txasdavB2,urn:epc:id:sgln:71asdjpaksjdp73736.79084.0,2026-09-22 03:41:05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,19c728348db858d867dd646a0b770086d6995375528adf...,ACasaDC,BG01,missing_location,28,LEBGR1e003Tp4N3J,LEBGR1f005OF8asda4ya,4276,hQsJAyxU94asldka;lsdkKFfK9,urn:epc:id:sgln:1233367.11754.0,2026-09-22 03:41:05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,bafd246a473e8a912a681d44341c53fa4e7cdfa234e892...,ACasaDC,BG02,missing_location,29,LEBGR1e003Tasdp4N3J,,2745,8GrI42lasdooka;sldHnslSc05,urn:epc:id:sgln:1233367.10054.0,2026-09-22 03:41:05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,829884cf2ecc58df7e752e89b81b9e1ac3eb32b254c1d1...,ACasdasDC,SGLN_mapping,complete,1,LEWL1EQwJwW9,LEWL1Fjwo8Cb,6025,Zsv77YiBg_oBtXQ,urn:epc:id:sgln:2143135.29114.0,2026-09-22 03:37:22,1.0,Ang Mo Kio Distribution Centre,Warehouse,SG-AMK-01,SG,Singapore,569880,2026-09-24 08:11:05
9,ef04da778a75f17ea5b6b690c698d7974edae1ce7f8f70...,ACasdqwas,SGLN_mapqweqwping,missing_location,26,LEWL1EQwJwW9,LEWL1Fjwo8Cb,6025,Zsv77YiBg_oBtXQ,urn:epc:id:sgln:2143135.29114.0,2026-09-22 03:41:05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
# 2. latest outcome per record + rule (rows come back oldest first)
results = pd.DataFrame(requests.get(f"{BASE_URL}/entities/validation-results").json())

latest = results.groupby(["businessEntityCode", "rule_id"], as_index=False).last()
display(latest.pivot(index="businessEntityCode", columns="rule_id", values="passed"))


rule_id,API-ROMANIZE-NAME,JOIN-COMPLETENESS,NB2-ENTITY-EOID,NB2-LOCATION-COUNTRY
businessEntityCode,,,,
BE-001,1.0,1.0,1.0,1.0
BE-002,1.0,1.0,1.0,1.0
BE-003,1.0,0.0,1.0,0.0
BE-999,1.0,0.0,0.0,1.0
BG01,1.0,NaN,NaN,NaN
BG02,1.0,NaN,NaN,NaN
BGN1,1.0,NaN,NaN,NaN
DOCS99,1.0,NaN,NaN,NaN
SGLN_mapping,1.0,NaN,NaN,NaN


In [16]:
# 3. every row, newest first
display(results.iloc[::-1][
    ["businessEntityCode", "source", "rule_id", "passed", "status", "description"]
])


,businessEntityCode,source,rule_id,passed,status,description
84,BE-999,location,NB2-LOCATION-COUNTRY,1,NaN,NaN
83,BE-999,location,NB2-ENTITY-EOID,0,-21.0,EOID is missing
82,BE-999,location,API-ROMANIZE-NAME,1,NaN,NaN
81,BE-999,location,JOIN-COMPLETENESS,0,-11.0,entity data is missing for this systemCode + b...
80,BE-002,location,NB2-LOCATION-COUNTRY,1,NaN,NaN
...,...,...,...,...,...,...
4,BGN1,NaN,API-ROMANIZE-NAME,1,NaN,NaN
3,BG02,NaN,API-ROMANIZE-NAME,1,NaN,NaN
2,BG01,NaN,API-ROMANIZE-NAME,1,NaN,NaN
1,DOCS99,NaN,API-ROMANIZE-NAME,1,NaN,NaN


In [21]:
results[~results["description"].isnull()]

,entity_hash,systemCode,businessEntityCode,source,entity_id,location_id,rule_id,passed,severity,status,description,created_at
45,668bae11288e4ca931d34fa811bf20bb7870342e124c98...,SYS-A,BE-001,location,NaN,21.0,JOIN-COMPLETENESS,0,Warning,-11.0,entity data is missing for this systemCode + b...,2026-09-24 08:54:29
47,668bae11288e4ca931d34fa811bf20bb7870342e124c98...,SYS-A,BE-001,location,NaN,21.0,NB2-ENTITY-EOID,0,Critical,-21.0,EOID is missing,2026-09-24 08:54:29
53,28755534b8b0ba9405dc6740f02b37abc10a6483124d3a...,SYS-A,BE-002,entity,47.0,NaN,JOIN-COMPLETENESS,0,Warning,-12.0,location data is missing for this systemCode +...,2026-09-24 08:54:52
56,28755534b8b0ba9405dc6740f02b37abc10a6483124d3a...,SYS-A,BE-002,entity,47.0,NaN,NB2-LOCATION-COUNTRY,0,Critical,-22.0,country is missing,2026-09-24 08:54:52
57,45d93593090845f9d17f8bbc3022f76554f03fb43e33dc...,SYS-A,BE-003,entity,48.0,NaN,JOIN-COMPLETENESS,0,Warning,-12.0,location data is missing for this systemCode +...,2026-09-24 08:54:52
60,45d93593090845f9d17f8bbc3022f76554f03fb43e33dc...,SYS-A,BE-003,entity,48.0,NaN,NB2-LOCATION-COUNTRY,0,Critical,-22.0,country is missing,2026-09-24 08:54:52
69,28755534b8b0ba9405dc6740f02b37abc10a6483124d3a...,SYS-A,BE-002,entity,50.0,NaN,JOIN-COMPLETENESS,0,Warning,-12.0,location data is missing for this systemCode +...,2026-09-25 03:46:32
72,28755534b8b0ba9405dc6740f02b37abc10a6483124d3a...,SYS-A,BE-002,entity,50.0,NaN,NB2-LOCATION-COUNTRY,0,Critical,-22.0,country is missing,2026-09-25 03:46:32
73,45d93593090845f9d17f8bbc3022f76554f03fb43e33dc...,SYS-A,BE-003,entity,51.0,NaN,JOIN-COMPLETENESS,0,Warning,-12.0,location data is missing for this systemCode +...,2026-09-25 03:46:32
76,45d93593090845f9d17f8bbc3022f76554f03fb43e33dc...,SYS-A,BE-003,entity,51.0,NaN,NB2-LOCATION-COUNTRY,0,Critical,-22.0,country is missing,2026-09-25 03:46:32


In [ ]:
"D/Python/Event Hub/entity_api"

### What to expect

| record | entity | location | JOIN-COMPLETENESS |
|---|---|---|---|
| BE-001 | step 4 | steps 3, 7 | fails -11 at step 3, passes afterwards |
| BE-002 | step 5 | step 6 | fails -12 at step 5, passes at step 6 |
| BE-003 | step 5 | step 7 | fails -12 at step 5, passes at step 7 |
| BE-004 | step 8 | — | fails -12 |
| BE-999 | step 8 | step 6 | fails -11 at step 6, passes at step 8 |

`NB2-ENTITY-EOID` fails while only the location side exists, and
`NB2-LOCATION-COUNTRY` fails while only the entity side exists — each rule's
field lives on the side that isn't there yet. Both pass once the record is
joined.
